<a href="https://colab.research.google.com/github/dxda6216/ttron2excel/blob/main/ttron_data_file_to_excel_file_20260526a1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import csv
import math
import numpy as np
from datetime import datetime, timedelta, timezone
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MultipleLocator
import io
from google.colab import files
from scipy import signal # Added for sinc filter implementation
import zipfile
import os

#@title Converting Taylortron TRACES file (TRACES.nnn) to Excel file {single-column: true}
#@markdown **This script works only with a specific format of data files (*TRACES.nnn* files) generated by the [Taylortron](https://doi.org/10.1080/09291018209359765) in the Carl Johnson Lab.**

#@markdown 1. Input the experiment number (avoid spaces and special characters).
#@markdown 2. Input the experiment title (this field can be blank).
#@markdown 3. Input the date on which the experiment started.
#@markdown 4. Select 'Sinc Filter' ([firwin](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html)) or '[Moving Average](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)' for detrending.
#@markdown 5. If 'Sinc Filter' is chosen, adjust 'Sinc_Filter_Cutoff_Period_Hours' and 'Sinc_Filter_Order'.
#@markdown 6. If 'Moving Average' is chosen, adjust 'Window_size_for_trend_line_moving_average'.
#@markdown 7. **Runtime** -> **Restart and run all** (or press **Ctrl+M** and then press **Ctrl+F9**)
#@markdown 8. Wait until `Choose Files` or `Browse...` button appears below.
#@markdown 9. Click `Choose Files` or `Browse...` button and select *TRACES.nnn* file in your computer.
#@markdown 10. Wait a while. two Excel files, one ZIP file, and one PDF file will be saved in "Downloads" folder in your computer.

#@markdown - The first Excel file will have multiple spreadsheets, conatining all the raw data, smoothed data, and detranded data.
#@markdown - The second Excel file will have a single spreadsheet, conatining only the raw time series data. The interval time will be indicated as a sheet name of the Excel spreadsheet. This file can be opened with data analysis programs such as [pyBOAT](https://github.com/tensionhead/pyBOAT).
#@markdown - The ZIP file will contain separate data files (.dat files) for each of the channels. The .dat files can be opened with the [LumiCycle](https://actimetrics.com/products/lumicycle/) Analysis program.

# --- User Parameters ---
Experiment_number = 'CYxxx' #@param {type:"string"}
Experiment_title = '' #@param {type:"string"}
Date_experiment_started = '2026-01-01' #@param {type:"date"}

Detrending_Method = "Sinc Filter" #@param ["Sinc Filter", "Moving Average"]

# Sinc Filter Parameters (only used if 'Sinc Filter' is selected)
Sinc_Filter_Cutoff_Period_Hours = 48 # @param {type:"slider", min:1, max:240, step:1}
Sinc_Filter_Order = 101 # @param {type:"slider", min:1, max:361, step:2}

# Moving Average Parameters (only used if 'Moving Average' is selected)
Window_size_for_trend_line_moving_average = 24 # @param {type:"slider", min:1, max:120, step:1}

Data_Plotting = "Plotting the channel 00 data last" #@param ["Plotting the channel 00 data first", "Plotting the channel 00 data last"]

# Setting the X-axis major tick interval
Major_Ticks = "Every 24 hours" #@param ["Every 12 hours", "Every 24 hours", "Every 48 hours"]

# Setting the X-axis minor tick interval
Minor_Ticks = "Every 12 hours" #@param ["No minor ticks", "Every 2 hours", "Every 4 hours", "Every 6 hours", "Every 12 hours", "Every 24 hours"]

# --- Constants and Mappings ---
major_ticks_map = {
    "Every 12 hours": 12,
    "Every 24 hours": 24,
    "Every 48 hours": 48
}
mjt = major_ticks_map.get(Major_Ticks)

minor_ticks_map = {
    "No minor ticks": 0,
    "Every 2 hours": 2,
    "Every 4 hours": 4,
    "Every 6 hours": 6,
    "Every 12 hours": 12,
    "Every 24 hours": 24
}
mit = minor_ticks_map.get(Minor_Ticks)

miton = False if mit == 0 or mit >= mjt else True

# --- Functions ---

# Cleaning up old files
def clean_up_old_files():
    print('Cleaning up old data files...')
    for ext in ['*.xlsx', '*.pdf', '*.dat', '*.zip', 'TRACES.*', 'Traces.*', 'traces.*']:
        os.system(f'rm -f {ext}')
    print('Cleanup complete.')

# Uploading a TRACE file
def upload_traces_file():
    print('\nUploading TRACES.nnn file...')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded. Please upload a TRACES.nnn file.")
    return next(iter(uploaded))

# Extracting the data from the TRACE file to a Pandas dataframe
def read_traces_data(ttronfilename):
    print('\nReading the data...')
    colnames = ["Hours","00","01","02","03","04","05","06","07","08","09","10","11","12","13","14","15","16","17","18","19","20","21","22","23","24","25","26","27","28","29"]

    df = pd.read_csv(ttronfilename, header=None, sep ='\t', skiprows=3, skipfooter=1, index_col=False, names=colnames, engine='python')
    df2 = df.iloc[:, 0:31]

    print("Raw DataFrame (first 5 rows):")
    print(df.head())

    number_of_rows = len(df.index)
    if number_of_rows == 0:
        raise ValueError("No data rows found in the TRACES file after skipping headers and footers.")

    last_row_index = number_of_rows - 1
    first_time_point = df.loc[0]['Hours']
    last_time_point = df.loc[last_row_index]['Hours']

    print(f'Number of rows: {number_of_rows}')
    print(f'First time points: {first_time_point} h')
    print(f'Last time points: {last_time_point} h')

    total_time = last_time_point - first_time_point
    total_time_in_days = total_time / 24.0
    print(f'Total time duration: {total_time} h = {total_time_in_days} days')

    time_interval = total_time / last_row_index if last_row_index > 0 else 0
    excel2_sheet_name = f'INTVL = {time_interval:.15f} h'
    print(f'Average time interval: {time_interval} h')

    return df, df2, number_of_rows, total_time, time_interval, excel2_sheet_name

# Calculating trend lines and Detrending dat
def calculate_trend_and_detrend(df, time_interval, detrending_method, number_of_rows, sinc_filter_cutoff_period_hours, sinc_filter_order, window_size_ma):
    print(f'\nCalculating trend line and detrended data using {detrending_method}...')

    df_5PMA = df.rolling(window=5, center=True, min_periods=1).mean()
    df_9PMA = df.rolling(window=9, center=True, min_periods=1).mean()

    df_TL = pd.DataFrame() # Trend Line DataFrame
    trendline_sheet_name = ''
    filter_order_actual = sinc_filter_order # To store the possibly adjusted filter order

    if detrending_method == "Moving Average":
        # Moving average time window for trend line
        tws = math.ceil(window_size_ma / time_interval)
        if tws % 2 == 0:
            tws += 1 # Ensure window size is odd

        twss = int(tws)
        twst = time_interval * (twss - 1)
        print(f'Window size for trend line (Moving Average): {twss} points ({twst:.6f} h)')
        df_TL = df.rolling(window=twss, center=True, min_periods=1).mean()
        # Fill NaN values at the boundaries for a complete trend line
        df_TL = df_TL.fillna(method='bfill').fillna(method='ffill')
        trendline_sheet_name = f'Trend line ({twss}PMA)'

    elif detrending_method == "Sinc Filter":
        sampling_rate = 1 / time_interval # samples per hour
        cutoff_frequency = 1 / sinc_filter_cutoff_period_hours # cycles per hour
        nyquist_frequency = 0.5 * sampling_rate
        norm_cutoff = cutoff_frequency / nyquist_frequency

        # Ensure filter order is not too large for filtfilt
        max_filter_order_for_data = int(number_of_rows / 3) - 1 # Rule of thumb for filtfilt stability
        filter_order_actual = sinc_filter_order

        if filter_order_actual >= max_filter_order_for_data:
            print(f"Warning: Sinc Filter Order ({filter_order_actual}) is too high for data length ({number_of_rows} points).")
            filter_order_actual = max_filter_order_for_data
            if filter_order_actual % 2 == 0: # Ensure filter order is odd
                filter_order_actual -= 1
            if filter_order_actual <= 0: # Ensure it's at least 1 if data is very short
                filter_order_actual = 1
            print(f"Adjusting Sinc Filter Order to {filter_order_actual} for stability with filtfilt.")

        if filter_order_actual % 2 == 0: # Ensure filter order is odd for filtfilt stability
            filter_order_actual += 1

        print(f"Sampling Rate: {sampling_rate:.2f} samples/hour")
        print(f"Cutoff Period for Sinc Filter: {sinc_filter_cutoff_period_hours} hours")
        print(f"Cutoff Frequency: {cutoff_frequency:.4f} cycles/hour")
        print(f"Normalized Cutoff Frequency: {norm_cutoff:.4f}")
        print(f"Sinc Filter Order: {filter_order_actual}")

        # Design the FIR filter (sinc-like filter)
        sinc_filter_coeffs = signal.firwin(filter_order_actual, norm_cutoff, pass_zero='lowpass')

        df_TL = df.copy() # Use df_TL for the trend output for consistency

        # Apply the filter to each channel to get the trend
        for k in range(0, 30, 1):
            channelnumber = str(k).zfill(2)
            if df[channelnumber].isnull().all(): # Skip if channel data is all NaN
                df_TL[channelnumber] = np.nan
                continue

            # Handle NaNs: interpolate or fill before filtering.
            data_to_filter = df[channelnumber].interpolate(method='linear', limit_direction='both', axis=0)

            try:
                df_TL[channelnumber] = signal.filtfilt(sinc_filter_coeffs, [1.0], data_to_filter)
            except ValueError as e:
                print(f"Warning: Could not apply sinc filter to channel {channelnumber}. Error: {e}")
                print("Consider reducing filter_order or increasing data length.")
                df_TL[channelnumber] = np.nan # Set to NaN if filtering fails

        trendline_sheet_name = f'Trend line (Sinc Filter C={sinc_filter_cutoff_period_hours}h O={filter_order_actual})'

    # Detrend the data by subtracting the trend line
    dtdf = df - df_TL
    dtdf['Hours'] = df_TL['Hours'] # Keep the 'Hours' column intact
    dtdf_5PMA = dtdf.rolling(window=5, center=True, min_periods=1).mean()
    dtdf_9PMA = dtdf.rolling(window=9, center=True, min_periods=1).mean()

    return df_5PMA, df_9PMA, df_TL, dtdf, dtdf_5PMA, dtdf_9PMA, trendline_sheet_name, filter_order_actual

# Generate Excel files from the dataframes
def generate_excel_outputs(df, df_5PMA, df_9PMA, df_TL, dtdf, dtdf_5PMA, dtdf_9PMA,
                           experiment_number, experiment_title, date_experiment_started,
                           ttronfilename, number_of_rows, total_time, time_interval,
                           detrending_method, trendline_sheet_name, sinc_filter_cutoff_period_hours,
                           sinc_filter_order_actual, window_size_ma, excel2_sheet_name):
    print('\nGenerating an Excel file containing all the data...')
    now = datetime.now(timezone.utc)
    processed_dnt_str = now.strftime("%Y-%m-%d %H:%M:%S")

    # Prepare metadata for the 'Note' sheet
    trend_param_info = ''
    if detrending_method == 'Moving Average':
        trend_param_info = f'{window_size_ma}h window'
    elif detrending_method == 'Sinc Filter':
        trend_param_info = f'{sinc_filter_cutoff_period_hours}h cutoff, {sinc_filter_order_actual} order'

    note_data = {
        'A': ['Experiment Number', 'Experiment Title', 'Experiment Start Date', 'TRACES File', '', 'Number of Time Points', 'Total Time Duration (Hours)', 'Average Time Interval (Hours)', 'Detrending Method', 'Trend Line Parameter', '', 'Data Processed Date and Time (UTC)'],
        'B': ['', '', '', '', '', '', '', '', '', '', '', ''],
        'C': ['', '', '', '', '', '', '', '', '', '', '', ''],
        'D': ['', '', '', '', '', '', '', '', '', '', '', ''],
        'E': [experiment_number, experiment_title, date_experiment_started, ttronfilename, '', number_of_rows, total_time, time_interval, detrending_method, trend_param_info, '', processed_dnt_str]
    }
    note_df = pd.DataFrame.from_dict(note_data)

    output_excel_filename = f"{experiment_number}_data.xlsx"
    with pd.ExcelWriter(output_excel_filename) as writer:
        note_df.to_excel(writer, sheet_name='Note', index=None, header=False)
        df.to_excel(writer, sheet_name='Raw Data', index=False)
        df_5PMA.to_excel(writer, sheet_name='5-point moving average (5PMA)', index=False)
        df_9PMA.to_excel(writer, sheet_name='9-point moving average (9PMA)', index=False)
        df_TL.to_excel(writer, sheet_name=trendline_sheet_name, index=False)
        dtdf.to_excel(writer, sheet_name='Detrended Data', index=False)
        dtdf_5PMA.to_excel(writer, sheet_name='Detrended Data 5PMA', index=False)
        dtdf_9PMA.to_excel(writer, sheet_name='Detrended Data 9PMA', index=False)

        for k in range(0, 30, 1):
            channelnumber = str(k).zfill(2)
            chnum = f'Channel {channelnumber}'
            dfx = pd.DataFrame()
            dfx['Hours'] = df['Hours']
            dfx['Raw_data'] = df[channelnumber]
            dfx['5PMA'] = df_5PMA[channelnumber]
            dfx['9PMA'] = df_9PMA[channelnumber]
            dfx['trend_line'] = df_TL[channelnumber]
            dfx['detrended_data'] = dtdf[channelnumber]
            dfx['detrended_data_5PMA'] = dtdf_5PMA[channelnumber]
            dfx['detrended_data_9MPA'] = dtdf_9PMA[channelnumber]
            dfx.to_excel(writer, sheet_name=chnum, index=False)

    print(f'\nExcel file: {output_excel_filename} has been generated.')

    print('\nGenerating an Excel file containing only the raw data...')
    output_excel_filename2 = f"{experiment_number}_data_2.xlsx"
    with pd.ExcelWriter(output_excel_filename2) as writer:
        df.iloc[:, 0:31].to_excel(writer, sheet_name=excel2_sheet_name, index=False, header=True)
    print(f'\nExcel file: {output_excel_filename2} has been generated.')

    return output_excel_filename, output_excel_filename2

# Generating .dat file and compressing them into a .zip file
def generate_dat_zip_files(df, experiment_number):
    print('\nGenerating a data file for each channel (.dat files)...')
    df_days = df.copy()
    df_days['Days'] = df_days['Hours'] / 24.000 # Create a copy to avoid modifying original df

    dat_files = []
    for k in range(0, 30, 1):
        channelnumber = str(k).zfill(2)
        datfilename = f'{channelnumber}.dat'
        df_days.to_csv(datfilename, header=False, index=False, sep ='\t', columns=['Days', channelnumber])
        dat_files.append(datfilename)

    print('\nPacking .dat files into a zip file...')
    zip_output_filename = f'{experiment_number}_data.zip'
    with zipfile.ZipFile(zip_output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in dat_files:
            zipf.write(file)
            os.remove(file) # Clean up individual .dat files after zipping

    print(f'\nZip file: {zip_output_filename} has been generated.')
    return zip_output_filename

# Plotting the data
def generate_all_plots(df, df_TL, dtdf, dtdf_5PMA, experiment_number, detrending_method,
                         sinc_filter_cutoff_period_hours, filter_order_actual,
                         data_plotting_choice, mjt, miton, mit):
    print('\nGenerating plots...')
    plot_output_pdf = f"{experiment_number}_data_plots.pdf"
    pp = PdfPages(plot_output_pdf)

    # Determine channel list based on plotting choice
    if data_plotting_choice == "Plotting the channel 00 data first":
        chlist = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29]
    else:
        chlist = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,0]

    x = df['Hours']
    x_scale_min = int(math.floor(min(x)*(1/24)))*24
    x_scale_max = int(math.ceil(max(x)*(1/12)))*12+12
    xtickslist = list(range(x_scale_min, x_scale_max, mjt))

    # --- Plotting Raw Data with Trend Line (30 channels in one figure) ---
    plt.rcParams.update({'figure.max_open_warning': 0})
    fig1 = plt.figure(figsize=(11, 8.5))
    fig1.subplots_adjust(hspace=0.15)
    fig1.suptitle(experiment_number, fontsize=12)

    plt.rc('font', size=5)
    plt.rc('axes', titlesize=4)
    plt.rc('axes', labelsize=4)
    plt.rc('xtick', labelsize=4)
    plt.rc('ytick', labelsize=4)
    plt.rc('legend', fontsize=3)
    plt.rc('figure', titlesize=5)

    subplotnumber = 1
    for k in chlist:
        channelnumber = str(k).zfill(2)
        ax = fig1.add_subplot(10, 3, subplotnumber)
        y = df[channelnumber]
        y_cma = df_TL[channelnumber]

        print(f'Plotting Channel {channelnumber} (Raw + Trend Line)')
        chlabel = f'Ch # {channelnumber}'
        ax.scatter(x, y, s=0.1, c='blue', label=chlabel)
        ax.plot(x, y_cma, '-r', linewidth=0.5, label=f'trend line ({detrending_method})')
        ax.set_xlim(x_scale_min - 6, x_scale_max)
        y_scale_max = int(max(y) * 1.100) if not y.empty and not y.isnull().all() else 100 # Default if y is empty/all NaN
        ax.set_ylim(0, y_scale_max)
        ax.set_xticks(xtickslist)
        if miton: ax.xaxis.set_minor_locator(MultipleLocator(mit))
        ax.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
        ax.legend(loc='upper right', fontsize=4)
        subplotnumber += 1

    fig1.text(0.50, 0.06, 'Time (hours)', horizontalalignment='center', fontsize=10)
    fig1.text(0.08, 0.50, 'Bioluminescence', horizontalalignment='center', verticalalignment='center', rotation='vertical', fontsize=10)
    pp.savefig(fig1)
    plt.close(fig1)

    # --- Plotting Detrended Data with Smoothed Line (30 channels in one figure) ---
    fig2 = plt.figure(figsize=(11, 8.5))
    fig2.subplots_adjust(hspace=0.15)
    fig2.suptitle(f'{experiment_number} - detrended data ({detrending_method})', fontsize=12)

    subplotnumber = 1
    for k in chlist:
        channelnumber = str(k).zfill(2)
        ax = fig2.add_subplot(10, 3, subplotnumber)
        y_dt = dtdf[channelnumber]
        y_dt_cma = dtdf_5PMA[channelnumber]

        chlabel = f'Ch # {channelnumber} - detrend'
        print(f'Plotting Channel {channelnumber} (Detrended)')
        ax.scatter(x, y_dt, s=0.1, c='blue', label=chlabel)
        ax.plot(x, y_dt_cma, '-r', linewidth=0.5)
        ax.set_xlim(x_scale_min - 6, x_scale_max)
        ax.set_xticks(xtickslist)
        if miton: ax.xaxis.set_minor_locator(MultipleLocator(mit))
        ax.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
        ax.legend(loc='upper right', fontsize=4)
        subplotnumber += 1

    fig2.text(0.50, 0.06, 'Time (hours)', horizontalalignment='center', fontsize=10)
    fig2.text(0.08, 0.50, 'Detrended Bioluminescence', horizontalalignment='center', verticalalignment='center', rotation='vertical', fontsize=10)
    pp.savefig(fig2)
    plt.close(fig2)

    # --- Plotting Individual Raw Data with 5-PMA Smoothed Line (one channel per figure) ---
    plt.rc('font', size=10)
    plt.rc('axes', titlesize=10)
    plt.rc('axes', labelsize=10)
    plt.rc('xtick', labelsize=9)
    plt.rc('ytick', labelsize=9)
    plt.rc('legend', fontsize=6)
    plt.rc('figure', titlesize=14)

    for k in chlist:
        channelnumber = str(k).zfill(2)
        fig3 = plt.figure(figsize=(11, 8.5))
        fig3.subplots_adjust(hspace=0.15)
        fig3.suptitle(f'{experiment_number} Ch # {channelnumber}', fontsize=14)
        ax = fig3.add_subplot(1, 1, 1)

        y = df[channelnumber]
        y_5cma = df_5PMA[channelnumber]
        print(f'Plotting Channel {channelnumber} (Individual Raw + 5PMA)')
        ax.scatter(x, y, s=5.0, c='violet', label='Bioluminescence')
        ax.plot(x, y_5cma, '-r', linewidth=1.0, label='Smoothed line (5-point moving average, centered)')
        ax.set_xlim(x_scale_min, x_scale_max)
        y_scale_max = int(max(y) * 1.100) if not y.empty and not y.isnull().all() else 100
        ax.set_ylim(0, y_scale_max)
        ax.set_xticks(xtickslist)
        if miton: ax.xaxis.set_minor_locator(MultipleLocator(mit))
        ax.set_xlabel('Hours', fontsize=12)
        ax.set_ylabel('Bioluminescence', fontsize=12)
        ax.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
        ax.legend(loc='upper right', fontsize=6)
        pp.savefig(fig3)
        plt.close(fig3)

    # --- Plotting Individual Raw and Detrended Data (one channel per figure, two subplots) ---
    for k in chlist:
        channelnumber = str(k).zfill(2)
        fig4 = plt.figure(figsize=(11, 8.5))
        fig4.suptitle(f'{experiment_number} Ch # {channelnumber}', fontsize=14)

        y = df[channelnumber]
        y_cma = df_TL[channelnumber]
        y_dt = dtdf[channelnumber]
        y_dt_cma = dtdf_5PMA[channelnumber]

        ax1 = fig4.add_subplot(2, 1, 1)
        print(f'Plotting Channel {channelnumber} (Raw + Detrended, Two-subplot view)')
        ax1.scatter(x, y, s=5.0, c='violet', label='Bioluminescence')

        if detrending_method == "Moving Average":
            ax1.plot(x, y_cma, '-r', linewidth=1.0, label=f'Trend line ({window_size_ma}-point moving average, centered)')
        else: # Sinc Filter
            ax1.plot(x, y_cma, '-r', linewidth=1.0, label=f'Trend line (Sinc Filter C={sinc_filter_cutoff_period_hours}h O={filter_order_actual})')

        ax1.set_xlim(x_scale_min, x_scale_max)
        y_scale_max = int(max(y) * 1.100) if not y.empty and not y.isnull().all() else 100
        ax1.set_ylim(0, y_scale_max)
        ax1.set_xticks(xtickslist)
        if miton: ax1.xaxis.set_minor_locator(MultipleLocator(mit))
        ax1.set_xlabel('Hours', fontsize=12)
        ax1.set_ylabel('Bioluminescence', fontsize=12)
        ax1.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
        ax1.legend(loc='upper right', fontsize=6)

        ax2 = fig4.add_subplot(2, 1, 2)
        print(f'Plotting Channel {channelnumber} (Detrended, Two-subplot view)')
        ax2.scatter(x, y_dt, s=5.0, c='violet', label='Detrended bioluminescence')
        ax2.plot(x, y_dt_cma, '-b', linewidth=1.0, label='Smoothed line (5-point moving average, centered)')
        ax2.set_xlim(x_scale_min, x_scale_max)
        ax2.set_xticks(xtickslist)
        if miton: ax2.xaxis.set_minor_locator(MultipleLocator(mit))
        ax2.set_xlabel('Hours', fontsize=12)
        ax2.set_ylabel('Detrended bioluminescence', fontsize=12)
        ax2.grid(True, linewidth=0.5, color='lightgray', linestyle='--')
        ax2.legend(loc='upper right', fontsize=6)

        pp.savefig(fig4)
        plt.close(fig4)

    pp.close()
    print(f'\nPDF file: {plot_output_pdf} has been generated.')
    return plot_output_pdf

# --- Main Execution ---
def main():
    global df, df_5PMA, df_9PMA, df_TL, dtdf, dtdf_5PMA, dtdf_9PMA
    clean_up_old_files()
    ttronfilename = upload_traces_file()

    df, df2, number_of_rows, total_time, time_interval, excel2_sheet_name = read_traces_data(ttronfilename)

    df_5PMA, df_9PMA, df_TL, dtdf, dtdf_5PMA, dtdf_9PMA, trendline_sheet_name, filter_order_actual = calculate_trend_and_detrend(
        df.copy(), time_interval, Detrending_Method, number_of_rows,
        Sinc_Filter_Cutoff_Period_Hours, Sinc_Filter_Order, Window_size_for_trend_line_moving_average
    )

    output_excel_filename, output_excel_filename2 = generate_excel_outputs(
        df, df_5PMA, df_9PMA, df_TL, dtdf, dtdf_5PMA, dtdf_9PMA,
        Experiment_number, Experiment_title, Date_experiment_started,
        ttronfilename, number_of_rows, total_time, time_interval,
        Detrending_Method, trendline_sheet_name, Sinc_Filter_Cutoff_Period_Hours,
        filter_order_actual, Window_size_for_trend_line_moving_average, excel2_sheet_name
    )

    zip_output_filename = generate_dat_zip_files(df.copy(), Experiment_number)

    plot_output_pdf = generate_all_plots(
        df.copy(), df_TL.copy(), dtdf.copy(), dtdf_5PMA.copy(),
        Experiment_number, Detrending_Method, Sinc_Filter_Cutoff_Period_Hours,
        filter_order_actual, Data_Plotting, mjt, miton, mit
    )

    # Download all generated files
    print('\nDownloading generated files...')
    files.download(output_excel_filename)
    files.download(output_excel_filename2)
    files.download(plot_output_pdf)
    files.download(zip_output_filename)
    print('\nAll files downloaded.')

if __name__ == '__main__':
    main()

# --- End of script ---

Cleaning up old data files...
Cleanup complete.

Uploading TRACES.nnn file...


Saving TRACES.835 to TRACES.835

Reading the data...
Raw DataFrame (first 5 rows):
   Hours      00     01     02     03     04     05     06     07     08  ...  \
0   0.00  437472  45131  44686  51450  45280  38194  56652  47355  51557  ...   
1   0.77  427229  32654  35907  42241  39229  35113  53362  45600  49717  ...   
2   1.53  430022  26476  31187  37096  37130  33704  50659  44752  48766  ...   
3   2.32  429442  24935  29054  33724  36143  33515  47622  45178  47841  ...   
4   3.08  429714  26997  30363  33395  37016  34797  45961  46958  48993  ...   

      20     21     22     23     24     25     26     27     28     29  
0  45131  40299  40270  39856  45467  38792  34989  36788  31137  35499  
1  48450  43172  42974  43214  48557  43317  39002  40496  34171  40097  
2  50596  44923  45451  45993  51590  46344  42417  42905  37560  44401  
3  52700  45851  47774  48289  52910  49887  45475  45458  41344  46769  
4  54920  46852  50629  50474  54598  52859  47630  47064  4

/usr/local/lib/python3.12/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")



Excel file: CYxxx_data.xlsx has been generated.

Generating an Excel file containing only the raw data...

Excel file: CYxxx_data_2.xlsx has been generated.

Generating a data file for each channel (.dat files)...

Packing .dat files into a zip file...

Zip file: CYxxx_data.zip has been generated.

Generating plots...
Plotting Channel 01 (Raw + Trend Line)
Plotting Channel 02 (Raw + Trend Line)
Plotting Channel 03 (Raw + Trend Line)
Plotting Channel 04 (Raw + Trend Line)
Plotting Channel 05 (Raw + Trend Line)
Plotting Channel 06 (Raw + Trend Line)
Plotting Channel 07 (Raw + Trend Line)
Plotting Channel 08 (Raw + Trend Line)
Plotting Channel 09 (Raw + Trend Line)
Plotting Channel 10 (Raw + Trend Line)
Plotting Channel 11 (Raw + Trend Line)
Plotting Channel 12 (Raw + Trend Line)
Plotting Channel 13 (Raw + Trend Line)
Plotting Channel 14 (Raw + Trend Line)
Plotting Channel 15 (Raw + Trend Line)
Plotting Channel 16 (Raw + Trend Line)
Plotting Channel 17 (Raw + Trend Line)
Plotting Channel

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All files downloaded.
